# MBPP EDA

- MBPP (Mostly Basic Python Programming)
- 출처 : https://huggingface.co/datasets/google-research-datasets/mbpp   
- 설명 : 구글 리서치(Google Research)에서 2021년에 발표한 Program Synthesis분야 데이터로, 약 1,000개의 크라우드소싱된 Python 프로그래밍 문제로 구성되어 있으며, 각 문제는 문제 설명, 코드 솔루션, 테스트 케이스를 포함하고 있다.
- 평가지표 : pass@1

## overview :
- ch01 : 데이터 톺아보기
- ch02 : task 활용 관점으로 데이터 분석

---
## ch01 : 데이터 톺아보기   

01. 데이터 확인
02. 한 샘플 자세히 보기
03. **동작 흐름 관점 분석**
04. 연구 관점 정리

### 1. 데이터 확인하기

In [1]:
# 1. 데이터 확인
from pathlib import Path
import json
import pandas as pd

DATA_PATH = Path("../../datasets/mbpp/raw/mbpp.jsonl")

rows = []
with open(DATA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

df = pd.DataFrame(rows)
df.head()

,task_id,text,code,test_list,test_setup_code,challenge_test_list
0,601,Write a function to find the longest chain whi...,"class Pair(object): \r\n\tdef __init__(self, a...","[assert max_chain_length([Pair(5, 24), Pair(15...",,[]
1,602,Write a python function to find the first repe...,"def first_repeated_char(str1):\r\n for index,...","[assert first_repeated_char(""abcabc"") == ""a"", ...",,[]
2,603,Write a function to get a lucid number smaller...,def get_ludic(n):\r\n\tludics = []\r\n\tfor i ...,"[assert get_ludic(10) == [1, 2, 3, 5, 7], asse...",,[]
3,604,Write a function to reverse words in a given s...,def reverse_words(s):\r\n return ' '.jo...,"[assert reverse_words(""python program"")==(""pro...",,[]
4,605,Write a function to check if the given integer...,def prime_num(num):\r\n if num >=1:\r\n for...,"[assert prime_num(13)==True, assert prime_num(...",,[]


In [2]:
print("shape:", df.shape)
print("columns:", df.columns.tolist())
df.dtypes

shape: (374, 6)
columns: ['task_id', 'text', 'code', 'test_list', 'test_setup_code', 'challenge_test_list']


task_id                 int64
text                   object
code                   object
test_list              object
test_setup_code        object
challenge_test_list    object
dtype: object

### 1-2. 데이터 필드 확인하기   
출처 : https://huggingface.co/datasets/google-research-datasets/mbpp  

> Data Fields:   
> - source_file: 데이터 출처
> - text/prompt: 모델에 입력되는 문제 설명
> - code: 정답 코드(참고용)
> - test_setup_code/test_imports: 테스트 실행 전에 피료한 코드(라이브러리, 환경 설정 등)
> - test_list: 정답 여부를 판단하는 테스트 코드 리스트(검증 조건)
> - challenge_test_list: 더 어려운 검증용 테스트(robustness)

### 2. 샘플 데이터 자세히 뜯어보기

In [3]:
# 샘플 1개에 대해 확인하기
sample = df.iloc[0].to_dict()
for k, v in sample.items():
    print(f"\n===== {k} =====")
    print(v if isinstance(v, str) else repr(v))


===== task_id =====
601

===== text =====
Write a function to find the longest chain which can be formed from the given set of pairs.

===== code =====
class Pair(object): 
	def __init__(self, a, b): 
		self.a = a 
		self.b = b 
def max_chain_length(arr, n): 
	max = 0
	mcl = [1 for i in range(n)] 
	for i in range(1, n): 
		for j in range(0, i): 
			if (arr[i].a > arr[j].b and
				mcl[i] < mcl[j] + 1): 
				mcl[i] = mcl[j] + 1
	for i in range(n): 
		if (max < mcl[i]): 
			max = mcl[i] 
	return max

===== test_list =====
['assert max_chain_length([Pair(5, 24), Pair(15, 25),Pair(27, 40), Pair(50, 60)], 4) == 3', 'assert max_chain_length([Pair(1, 2), Pair(3, 4),Pair(5, 6), Pair(7, 8)], 4) == 4', 'assert max_chain_length([Pair(19, 10), Pair(11, 12),Pair(13, 14), Pair(15, 16), Pair(31, 54)], 5) == 5']

===== test_setup_code =====


===== challenge_test_list =====
[]


### 3. 동작 흐름 관점 분석

한 샘플의 동작 :    
(1) input : text  

(2) output code : 모델이 함수 전체 생성  
    → code = 함수 이름, 파라미터, 클래스, 로직을 다 생성해야함  

(3) exec(code)   ---------------------- **Structural failure**   
    → 함수 생성

(4) exec(test_setup_code)  
    → check 함수 생성(import 등 환경 설정)  

(5) test_list 실행   ---------------------- **Semantic failure**   
    → for t in test_list: exec(t)


(1) input : text   
**모델 입력값 :  자연어 입력** 

In [4]:
# ===== text =====
# Write a function to find the longest chain which can be formed from the given set of pairs.

### 4. 연구 포인트

MBPP는 '코드를 생성하는 문제'가 아니라 자연어로 입력했을 때 '실행 가능한 코드를 생성해서 테스트를 통과하는 문제'로 보임

---
## ch02 : 데이터 톺아보기   

01. 전체 길이 분석
02. input/output 관점
03. Test 구조 분석
04. Research implication 

### 1. 길이 관점

In [5]:
# text_len : 모델에 들어가는 길이
# code_len : 모델이 출력하는 길이 관점 추정
# test_list_len : 평가 복잡도? 검증 조건의 길이

# 길이 (문자 기준)
for col in ["text", "code", "test_list", "test_setup_code", "challenge_test_list"]:
    df[f"{col}_len"] = df[col].astype(str).str.len()

# 개수 (리스트 기준)
df["test_list_count"] = df["test_list"].apply(len)
df["test_setup_code_count"] = df["test_setup_code"].apply(len)
df["challenge_test_count"] = df["challenge_test_list"].apply(len)

# 확인
df[[
    "text_len",
    "code_len",
    "test_list_len",
    "test_list_count",
    'test_setup_code_len',
    'test_setup_code_count',
    'challenge_test_list_len',
    "challenge_test_count"
]].describe()

,text_len,code_len,test_list_len,test_list_count,test_setup_code_len,test_setup_code_count,challenge_test_list_len,challenge_test_count
count,374.000000,374.000000,374.00000,374.0,374.000000,374.000000,374.0,374.0
mean,78.430481,178.382353,191.92246,3.0,1.451872,1.451872,2.0,0.0
std,21.132525,119.240945,104.80592,0.0,28.077862,28.077862,0.0,0.0
min,38.000000,30.000000,71.00000,3.0,0.000000,0.000000,2.0,0.0
25%,64.000000,99.000000,121.00000,3.0,0.000000,0.000000,2.0,0.0
50%,77.500000,144.000000,159.50000,3.0,0.000000,0.000000,2.0,0.0
75%,90.000000,209.750000,221.75000,3.0,0.000000,0.000000,2.0,0.0
max,249.000000,908.000000,816.00000,3.0,543.000000,543.000000,2.0,0.0


In [6]:
preview_cols = ['task_id', 'text', 'code', 'test_list', 'test_setup_code', 'challenge_test_list']
df[preview_cols].head(5)

,task_id,text,code,test_list,test_setup_code,challenge_test_list
0,601,Write a function to find the longest chain whi...,"class Pair(object): \r\n\tdef __init__(self, a...","[assert max_chain_length([Pair(5, 24), Pair(15...",,[]
1,602,Write a python function to find the first repe...,"def first_repeated_char(str1):\r\n for index,...","[assert first_repeated_char(""abcabc"") == ""a"", ...",,[]
2,603,Write a function to get a lucid number smaller...,def get_ludic(n):\r\n\tludics = []\r\n\tfor i ...,"[assert get_ludic(10) == [1, 2, 3, 5, 7], asse...",,[]
3,604,Write a function to reverse words in a given s...,def reverse_words(s):\r\n return ' '.jo...,"[assert reverse_words(""python program"")==(""pro...",,[]
4,605,Write a function to check if the given integer...,def prime_num(num):\r\n if num >=1:\r\n for...,"[assert prime_num(13)==True, assert prime_num(...",,[]


### 2. intput/output 구조 관찰

In [7]:
for i in range(3):
    print(f"\n================ SAMPLE {i} ================ len : {len(df.loc[i, 'text'])}\n")
    print(df.loc[i, "text"])


================ SAMPLE 0 ================ len : 91

Write a function to find the longest chain which can be formed from the given set of pairs.

================ SAMPLE 1 ================ len : 79

Write a python function to find the first repeated character in a given string.

================ SAMPLE 2 ================ len : 66

Write a function to get a lucid number smaller than or equal to n.


In [8]:
for i in range(3):
    print(f"\n================ SAMPLE {i} ================ len : {len(df.loc[i, 'code'])}\n")
    print(df.loc[i, "code"])


================ SAMPLE 0 ================ len : 364

class Pair(object): 
	def __init__(self, a, b): 
		self.a = a 
		self.b = b 
def max_chain_length(arr, n): 
	max = 0
	mcl = [1 for i in range(n)] 
	for i in range(1, n): 
		for j in range(0, i): 
			if (arr[i].a > arr[j].b and
				mcl[i] < mcl[j] + 1): 
				mcl[i] = mcl[j] + 1
	for i in range(n): 
		if (max < mcl[i]): 
			max = mcl[i] 
	return max

================ SAMPLE 1 ================ len : 136

def first_repeated_char(str1):
  for index,c in enumerate(str1):
    if str1[:index+1].count(c) > 1:
      return c 
  return "None"

================ SAMPLE 2 ================ len : 349

def get_ludic(n):
	ludics = []
	for i in range(1, n + 1):
		ludics.append(i)
	index = 1
	while(index != len(ludics)):
		first_ludic = ludics[index]
		remove_index = index + first_ludic
		while(remove_index < len(ludics)):
			ludics.remove(ludics[remove_index])
			remove_index = remove_index + first_ludic - 1
		index += 1
	return ludics


### 3. test 구조 관찰

In [ ]:
for i in range(3):
    print(f"\n================ TEST {i} - {df.loc[i, 'task_id']} ================ len : {len(df.loc[i, 'test_list'])}\n")
    print(df.loc[i, "test_list"])


================ TEST 0 - 601 ================ len : 3

assert max_chain_length([Pair(5, 24), Pair(15, 25),Pair(27, 40), Pair(50, 60)], 4) == 3

================ TEST 1 - 602 ================ len : 3

assert first_repeated_char("abcabc") == "a"

================ TEST 2 - 603 ================ len : 3

assert get_ludic(10) == [1, 2, 3, 5, 7]


### 4. 연구관점 정리

MBPP는 모델이 함수 구조와 로직을 동시에 생성해야 하기 때문에,HumanEval보다 structural generation burden이 더 크다.